# Neural Networks for Classification with MNIST

In [1]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset
from sklearn.datasets import load_digits

# For model and evaluation
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, ConfusionMatrixDisplay, log_loss

In [2]:
import torch.nn as nn

class NeuralNetwork(nn.Module):
    """
    A feedforward neural network for regression.

    Args:
        input_size (int): The size of the input features.
        hidden_layer_sizes (list of int): Sizes of hidden layers.
        activation_functions (list of callable): Activation functions for each layer.
        dropout_prob (float): Dropout probability.

    Raises:
        ValueError: If the number of activation functions doesn't match the layer sizes.

    Attributes:
        layers (nn.ModuleList): List of network layers.

    Example:
        To create a neural network with 2 hidden layers using ReLU activation and a dropout
        probability of 0.2:

        >>> network = NeuralNetwork(input_size=64, hidden_layer_sizes=[32, 16],
        ...                         activation_functions=[nn.ReLU, nn.ReLU],
        ...                         dropout_prob=0.2)
    """

    def __init__(self, output_size, input_size, hidden_layer_sizes, activation_functions, dropout_prob):
        """
        Initializes the NeuralNetwork.

        Args:
            output_size (int): The size of the output.
            input_size (int): The size of the input features.
            hidden_layer_sizes (list of int): Sizes of hidden layers.
            activation_functions (list of callable): Activation functions for each layer.
            dropout_prob (float): Dropout probability.

        Raises:
            ValueError: If the number of activation functions doesn't match the layer sizes.
        """


        if len(hidden_layer_sizes) + 1 > len(activation_functions):
            raise ValueError(f"Number of activation functions must be at least {len(hidden_layer_sizes)+1}.")
        
        super(NeuralNetwork, self).__init__()
        
        if len(hidden_layer_sizes) + 2 == len(activation_functions):
            self.last_activation = activation_functions[-1]
        else:
            self.last_activation = False
        
        layers = [input_size] + hidden_layer_sizes + [output_size]
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(layers[i], layers[i+1]),
                activation_functions[i](),
                nn.Dropout(p=dropout_prob)
            )
            for i in range(len(layers) - 1)
        ])

    def forward(self, x):
        """
        Forward pass through the neural network.

        Args:
            x (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Output prediction.
        """
        for layer in self.layers:
            x = layer(x)

        if self.last_activation!=False:
            x=self.last_activation(x)
        return x


In [3]:
class NeuralNetworkTrainer:
    """
    A class for training and evaluating neural network models.

    Attributes: 
        - activation_functions (list): Activation functions for each layer.
        - batch_size (int): Batch size for training.
        - criterion (torch.nn.Module): Loss function used for training.
        - dropout_prob (float): Dropout probability.
        - hidden_layer_sizes (list): Sizes of hidden layers.
        - learning_rate (float): Learning rate for optimization.
        - model (NeuralNetwork): The neural network model object.
        - num_epochs (int): Number of training epochs.
        - optimizer (torch.optim.Optimizer): Optimizer used for model optimization.
        - train_losses (list): Training set loss values per epoch.
        - train_acc (list): Training set Accuracy values per epoch.
        - val_losses (list): Validation set loss values per epoch.
        - val_acc (list): Validation set Accuracy values per epoch.

    Methods: 
        - `__init__(num_epochs=200, hidden_layer_sizes=[64,32], activation_functions=[nn.ReLU, nn.ReLU],
        learning_rate=0.001, batch_size=64, optimizer=optim.Adam, criterion=nn.MSELoss(), dropout_prob=0.1)`:
            Initialize the NeuralNetworkTrainer with hyperparameters and attributes.

        - `train(X_train, y_train, X_val, y_val, printups=False)`:
            Train the neural network model.

        - `suggested_epoch(threshold=1e-6, consecutive_count=5)`:
            Find the suggested number of epochs based on training and validation loss.

        - `validation_loss_graph()`:
            Plot loss for training and validation over epochs.

        - `validation_acc_graph()`:
            Plot Accuracy for training and validation over epochs.

        - `coefficient_heatmap()`:
            Visualize the neural network weights as heatmaps by layer.

        - `actual_vs_predicted(X_test, y_test, labels)`:
            Plot actual versus predicted values for test data.
    """
    
    def __init__(self, num_epochs=200, hidden_layer_sizes=[64,32], 
                 activation_functions=[nn.ReLU, nn.ReLU], learning_rate=0.001, 
                 batch_size=64, optimizer=optim.Adam,
                 criterion=nn.CrossEntropyLoss(), dropout_prob=0):
        """
        Initialize the NeuralNetworkTrainer with hyperparameters and attributes.

        Args:
            num_epochs (int): Number of training epochs (default is 200).
            hidden_layer_sizes (list): Sizes of hidden layers (default is [64, 32]).
            activation_functions (list): Activation functions for each layer (default is [nn.ReLU, nn.ReLU]).
            learning_rate (float): Learning rate for optimization (default is 0.001).
            batch_size (int): Batch size for training (default is 64).
            optimizer (torch.optim.Optimizer): Optimizer class for model optimization (default is optim.Adam).
            criterion (torch.nn.Module): Loss function for training (default is nn.CrossEntropyLoss()).
            dropout_prob (float): Dropout probability (default is 0.1).

        Raises:
            ValueError: If the number of activation functions does not match the number of hidden layers + 2.
        """
        
        # Hyperparameters
        self.num_epochs = num_epochs
        self.hidden_layer_sizes = hidden_layer_sizes
        self.activation_functions = activation_functions
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.optimizer = optimizer
        self.criterion = criterion
        self.dropout_prob = dropout_prob  # Specify dropout probability
        
        if len(self.activation_functions) < len(self.hidden_layer_sizes) + 1:
            print("WARNING: " + \
                  f"You should have at least {len(hidden_layer_sizes)+1} activation functions but " + \
                  f"you have {len(self.activation_functions)}. ")
        
        self.ideal_epochs = []

    def train(self, X_train, y_train, X_val, y_val, printups=False):
        """
        Train the neural network model.

        Args:
            X_train (torch.Tensor): Training input data.
            y_train (torch.Tensor): Training target data.
            X_val (torch.Tensor): Validation input data.
            y_val (torch.Tensor): Validation target data.
            printups (bool): If True, print training progress during epochs (default is False).
        """
        
        # Create DataLoaders for training and validation
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_dataset = TensorDataset(X_val, y_val)
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False)

        # Initialize the model, optimizer, and loss function
        output_size = torch.max(y_train) + 1
        self.model = NeuralNetwork(output_size, X_train.shape[1], self.hidden_layer_sizes, 
                                   self.activation_functions, self.dropout_prob)
        optimizer = self.optimizer(self.model.parameters(), lr=self.learning_rate)

        self.train_losses = []
        self.train_acc = []
        self.val_losses = []
        self.val_acc = []

        for epoch in range(self.num_epochs):
            
            self.model.train()
            train_loss = 0.0
            train_predictions = []
            train_labels = []
            
            correct = 0
            for inputs, labels in train_loader:
                optimizer.zero_grad()
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

                # Convert tensor to numpy array and flatten
                train_predictions.extend(outputs.argmax(1).numpy())
                train_labels.extend(labels.numpy())

            # Calculate training Accuracy
            train_acc = accuracy_score(train_labels, train_predictions)
            self.train_acc.append(train_acc)
            
            self.model.eval()
            val_loss = 0.0
            val_predictions = []
            val_labels = []
            
            with torch.no_grad():
                for inputs, labels in val_loader:
                    outputs = self.model(inputs)
                    loss = self.criterion(outputs, labels)
                    val_loss += loss.item()

                    # Convert tensor to numpy array and flatten
                    val_predictions.extend(outputs.argmax(1).numpy())
                    val_labels.extend(labels.numpy())

            # Calculate train and validation losses
            self.train_losses.append(train_loss / len(train_loader))
            self.val_losses.append(val_loss / len(val_loader))

            # Calculate validation Accuracy
            val_acc = accuracy_score(val_labels, val_predictions)
            self.val_acc.append(val_acc)

            if printups:
                print(f"Epoch [{epoch + 1}/{self.num_epochs}] - "
                      f"train dataset Loss: {self.train_losses[-1]:.4f}, "
                      f"train dataset Accuracy: {self.train_acc[-1]:.4f}, "
                      f"test or validation dataset Loss: {self.val_losses[-1]:.4f}", 
                      f"test or validation dataset Accuracy: {self.val_acc[-1]:.4f}")

    def suggested_epoch(self, threshold=1e-6, consecutive_count=5) -> int:
        """
        Find the index at which a list of values has leveled out by comparing consecutive differences.

        Args:
            threshold (float): The threshold to consider differences as "small" (default is 1e-6).
            consecutive_count (int): The number of consecutive values with small differences to be 
                                     considered "leveled out" (default is 5).

        Returns:
            int: The index where the validations losses have leveled out, or -1 if they don't.
        """
        
        if len(self.val_losses) < consecutive_count:
            return -1

        consecutive_small_diff_count = 0
        prev_value = self.val_losses[0]

        for i, value in enumerate(self.val_losses[1:], start=1):
            diff = abs(value - prev_value)
            if diff < threshold:
                consecutive_small_diff_count += 1
            else:
                consecutive_small_diff_count = 0

            if consecutive_small_diff_count >= consecutive_count:
                return i - consecutive_count + 1  # Return the index where leveling out started

            prev_value = value

        return -1  # Return -1 if values haven't leveled out

    def validation_loss_graph(self):
        """
        Plot loss for training and validation over epochs.
        """
        
        # Plot loss for training and validation
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, self.num_epochs + 1), self.train_losses, label='Train loss')
        plt.plot(range(1, self.num_epochs + 1), self.val_losses, label='Validation loss')
        if len(self.ideal_epochs) > 0:
            plt.axvline(x=self.ideal_epochs[0], color='r', linestyle='--', label=f'Epoch={self.ideal_epochs[0]}')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training and Validation loss vs. Epoch')
        plt.legend()
        plt.show()

    def validation_acc_graph(self):
        """
        Plot Accuracy for training and validation over epochs.
        """
        
        # Plot Accuracy for training and validation
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, self.num_epochs + 1), self.train_acc, label='Train Acc')
        plt.plot(range(1, self.num_epochs + 1), self.val_acc, label='Validation Acc')
        if len(self.ideal_epochs) > 0:
            plt.axvline(x=self.ideal_epochs[0], color='r', linestyle='--', label=f'Epoch={self.ideal_epochs[0]}')
        plt.xlabel('Epoch')
        plt.ylabel('Coefficient of Determination (Accuracy)')
        # plt.ylim(min(self.train_acc + self.val_acc),1)
        plt.title('Training and Validation Accuracy vs. Epoch')
        plt.legend()
        plt.show()

    def coefficient_heatmap(self):
        """
        Visualize the neural network weights as heatmaps by layer.
        """
       
        # Visualize the neural network weights as heatmaps (optional)
        sns.set(style="whitegrid")
        num_layers = len(self.hidden_layer_sizes)
       
        fig, axes = plt.subplots(1, num_layers, figsize=(4 * num_layers, 4))

        if type(axes) is not np.ndarray:
            axes=[axes]
       
        for i, ax in enumerate(axes):
            sns.heatmap(self.model.layers[i][0].weight.detach().numpy(), ax=ax, cmap="icefire", annot=False)
            ax.set_title(f'Hidden Layer {i+1} Weights')
           
        plt.show()

    def actual_vs_predicted(self, X_test, y_test, labels):
        """
        Plot actual versus predicted values.

        Args:
            X_test (torch.Tensor): Test input data.
            y_test (torch.Tensor): Test target data.
            labels: list of the labels.
        """

        # Make predictions on the test data
        self.model.eval()
        with torch.no_grad():
            y_pred = self.model(X_test).argmax(1).numpy()

        print(classification_report(y_test, y_pred, target_names=labels))
        
        disp1 = ConfusionMatrixDisplay.from_predictions(
                    y_test,
                    y_pred,
                    display_labels=labels,
                    normalize=None,
                    colorbar=False,
                )
        disp1.ax_.set_title('Confusion matrix')
        plt.grid(False)
        plt.show()


In [4]:
# Load digits dataset
digits = load_digits()
X = digits.data
y = digits.target
labels = digits.target_names
labels = [str(label) for label in labels]

# Standardize the features (important for neural networks)

# Split the dataset into training, validation, and testing sets

# Convert data to PyTorch tensors


In [5]:
# Set hyperparameters

# Creating the model

# Understanding the model


In [6]:
# Set epochs based on last model's analysis

# creating new model

# Understanding the model


# Write Up
Discuss your work and findings.